In [ ]:
from src.compute_text_representations import compute_text_representations
import pandas as pd
import torch
from src.utils import device, encode_df
from src.pairwise_dataset import PairwiseDataset
import numpy as np
from tqdm.auto import tqdm
import xarray as xr
import plotly.express as px
import scipy.stats as stats
from src.sampling_theory import *
import logging

# Set logging level to debug
logging.basicConfig(level=logging.DEBUG)

logger = logging.getLogger(__name__)

In [ ]:
device = "cpu"

In [ ]:
df = pd.read_csv("datasets/large.csv")
sentences = df["sentence"].tolist()
df = df.drop(columns="sentence")
embeddings = compute_text_representations(
    sentences, model_name="bert-base-uncased", token_aggregation="mean"
)
X = encode_df(df).to(device)
Y = embeddings[5].to(device)
dataset = PairwiseDataset(X, Y, n_pairs=512)

In [ ]:
corrs, _ = estimate_corrs(X, init_sample_size=32, max_margin=0.03)

In [ ]:
px.imshow(corrs)

In [ ]:
features = df.columns
x, y = torch.triu_indices(len(features), len(features))
feature_pairs = np.array(
    [f + ", " + f_ for f, f_ in zip(features[x], features[y])]
)

In [ ]:
dataset.n_pairs = 100 * 4096 * 2
X_batch, _ = next(iter(dataset))

In [ ]:
X_batch = X_batch.reshape(100, 4096 * 2, -1)

In [ ]:
corrs = batch_corrcoef(X_batch)

In [ ]:
cis = compute_ci(corrs.reshape(100, -1))